In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.bronze.supply_chain_orders")

drop_cols = ["customer_email", "customer_password", "product_image"]

int_cols = [
    "days_for_shipping_real", "days_for_shipment_scheduled", "late_delivery_risk",
    "category_id", "customer_id", "department_id", "order_customer_id", "order_id",
    "order_item_cardprod_id", "order_item_id", "order_item_quantity",
    "product_card_id", "product_category_id", "product_status",
]
money_cols = [
    "benefit_per_order", "sales_per_customer", "order_item_discount",
    "order_item_product_price", "sales", "order_item_total",
    "order_profit_per_order", "product_price",
]
double_cols = [
    "order_item_discount_rate", "order_item_profit_ratio", "latitude", "longitude",
]
ts_cols = ["order_date_dateorders", "shipping_date_dateorders"]

In [0]:
silver = bronze.drop(*drop_cols)

for c in int_cols:
    if c in silver.columns:
        silver = silver.withColumn(c, F.expr(f"try_cast({c} as bigint)"))
for c in money_cols:
    if c in silver.columns:
        silver = silver.withColumn(c, F.expr(f"try_cast({c} as decimal(18,2))"))
for c in double_cols:
    if c in silver.columns:
        silver = silver.withColumn(c, F.expr(f"try_cast({c} as double)"))
for c in ts_cols:
    if c in silver.columns:
        silver = silver.withColumn(c, F.expr(f"try_to_timestamp({c}, 'M/d/yyyy H:mm')"))

(silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.supply_chain_orders"))

In [0]:
s = spark.table("workspace.silver.supply_chain_orders")
print("rows:", s.count(), "| distinct order_item_id:", s.select("order_item_id").distinct().count())
s.select(
    F.count("*").alias("rows"),
    F.count("order_date_dateorders").alias("order_dates_parsed"),
    F.count("shipping_date_dateorders").alias("shipping_dates_parsed"),
).show()
s.printSchema()